# DEMONEXT ASCOM command/control Sandbox

This notebook is a sandbox for learning how to operate the DEMONEXT functions from Python 3 using ASCOM
and MaxIM DL Pro 7.

## STOP: Early Development Notebook!!!

This is an early development notebook used to develope the code that became the demonext module
and its associated classes.  

After 2025 Feb 1, use these notebooks instead
 * `StartUp_Shutdown.ipynb` - for lab system startup and shutdown
 * `TelescopeSandbox.ipynb` - to use the `Telescope` class for continued class development
 * `CameraSandbox.ipynb` - to use the `Camera` class for continued class development

## WARNING: READ THIS

**Never, Ever, Run All in this notebook at once**  It contains code that moves the telescope, and tests startup
and shutdown procedures. Use it piece-wise as it was made.  This is a notebook of experiments done on various
orders.

In [2]:
import sys
import os
import time
import math

# Windows Component Object Model (COM) client module.

from win32com.client import Dispatch

# modules we need from anaconda

import numpy as np

# FITS writing and handling

from astropy.io import fits

# astropy units, time, and coordinate functions go here

import astropy.units as u

# we use YAML for runtime configuration files

import yaml
from yaml import load
try:
    from yaml import CLoader as Loader
except ImportError:
    from yaml import Loader

## Convenience boolean and status code translations

Dictionaries to translate boolean and integer status codes returned by ASCOM properties into human-readable form.

In [5]:
# Boolean state convenience translation dictionaries

OnOff = {True:'On',False:'Off'}
YesNo = {True:'Yes',False:'No'}

# CCD camera status codes returned by the CCDCamera.CameraStatus property.
# Not all status codes are used/reported by all cameras.

camStatusCodes = {0:'Camera Not Connected',
                  1:'Camera reporting an error',
                  2:'Camera idle',
                  3:'Camera acquiring a Science image',
                  4:'Camera reading out',
                  5:'Camera writing to computer memory',
                  6:'Camera flushing sensor',
                  7:'Camera waiting for external trigger signal',
                  8:'Waiting for MaxIm to be ready to receive data',
                  9:'Camera waiting until time to acquire next image',
                  10:'Camera acquiring a Dark image using Simple Auto Dark',
                  11:'Camera acquiring a Bias image',
                  12:'Camera acquiring a Dark Image',
                  13:'Camera acquiring a Flat Image',
                  15:'Camera waiting on Filter Wheel'}

# Guide Camera calibration state codes returned by the CCDCamera.GuiderCalState property

guiderCalStates = {0:'Guide Camera Not Calibrated',
                   1:'Guide Camera Calibration in progress',
                   2:'Guide Camera Calibration Complete',
                   3:'Guide Camera Calibration Failed'}

## Instrument, Site, and Project information FITS header dictionaries

We save instrument, site, and project information for inclusion in image FITS headers in dictionaries using the
FITS keywords as the dictionary keys.  These are uploaded to augment the headers using the
`CCDCamera.SaveFITSKey()` method.

The instrument and observatory site dictionaries (`instInfo` and `siteInfo`) will eventually be populated by
reading a runtime configuration file so we can modify their contents without changing the source code. 
For these development exercises we hard code it.

The project information dictionary (`projInfo`) is a default placeholder that will get overloaded with 
specific project data later.

The primary use is to define all FITS header information that does not change unless there are major
changes to the instrument.  

The secondary use is to define a header template containing default entries that are expected to be overloaded
later. For example, the `projInfo` dictionary below is one such example: a prototye dictionary of FITS keywords
derived from project information that changes as we move among different projects executed during a given night.
It starts with generic values that would be appropriate for data taken during the daytime for testing or troubleshooting.

Finally, we aggregate these into a master `fitsHeader` dictionary that is used when acquiring data.  The
data acquisition functions are expected to *copy* `fitsHeader`, not modify it directly so we have a
safe default state.

### Method 1: Hard Coded defaults

Here we define the hardcoded defaults for the system.  These may be overloaded from the contents of a
configuration file formatted using YAML

In [8]:
# Instrument header data in dictionary form, key = FITS keyword, then value

instInfo = {'TELESCOP':'PlaneWave CDK20 f/6.8',
            'INSTRUME':'FLI PL23042 + CFW3-10',
            'FOCALLEN':3454.0,
            'APTDIA':0.508,
            'APTAREA':171854.9,
            'FOCUSER':'PlaneWave Hedrick',
            'FOCUSSSZ':1.0}

# Sierra Remote Observatories site information in dictionary form, key = FITS keyword, then value
# BEWARE: SITELAT and SITELONG are reserved keywords for MaxIm and require string arguments, use SITE_LON
# to avoid raising exceptions as we use decimal degrees.

siteInfo = {'OBSERVAT':'DEMONEXT Reboot',
            'SITENAME':'Sierra Remote Observatories',
            'SITE_LON':-119.41293,
            'SITE_LAT':37.07031,
            'SITE_EL':1405.0}

# Prototype project information dictionary: key = FITS keyword, then values
# These are placeholders for project information that comes from the scheduler

projInfo = {'PROJECT':'Engineering',
            'PI_NAME':'Engineering',
            'PI_INST':'OSU',
            'OBJECT':'Test Data',
            'OBSERVER':'DEMONEXT Observe-o-Bot'}

# Build a template supplementary FITS header dictionary: fitsHeader

fitsHeader = {}
fitsHeader.update(instInfo)
fitsHeader.update(siteInfo)
fitsHeader.update(projInfo)

### Method 2: Load info from a configuration file

We will use YAML to format our runtime configuration file so we don't have to hardcode the instrument and 
system setup info and can share a single config file or all elements of the system.

This is the preferred and sustainable way to flexibly configure the system without having to hack source
code.

In [11]:
configPath = "."
configFile = "demonext.txt"

cfgFile = f"{configPath}\\{configFile}"

if os.path.exists(cfgFile):
    with open(cfgFile,'r') as stream:
        try:
            config = yaml.load(stream,Loader=Loader)
        except Exception as exp:
            print(f"ERROR: could not open configuration file {cfgFile}: {exp}")
    
    siteInfo = config['site']
    instInfo = config['instrument']
    projInfo = config['project']
    pduInfo = config['pdu']
else:
    raise ValueError(f"Runtime configuration file {cfgFile} does not exist")

# build the template fitsHeader dictionary from the config file entries

fitsHeader = {}
fitsHeader.update(instInfo)
fitsHeader.update(siteInfo)
fitsHeader.update(projInfo)

print(pduInfo)

{'ipAddr': '140.254.79.215', 'userID': 'telemetry', 'passwd': 'D3monext', 'disableCert': True, 'outletNames': ['pc', 'ccd', 'telescope', '']}


## ASCOM Setup

We use the following ASCOM interfaces
 * `MaxIm.Application` - communicates with the MaxIM DL application class
 * `MaxIm.CCDCamera` - Operate the science and guide cameras through MaxIm DL Pro 7
 * `SiTech.Telescope` - Operate the telescope mount SiTech ServerController II via the PlaneWave STI app
 * `ASCOM.PWI3.Focuser` - Operate the PlaneWave Hedrick focuser (science camera) using the PlaneWave PWI3 app
 * `PlaneWave.AutoFocus` - access the PlaneWave telescope autofocus system through the PWI3 app

## Applications we need running

We run these applications, in order
 * `SitechExe.exe` - PlaneWave STI version 1.4.3, must execute by hand (not sure why, but we do), must run FIRST before MaxIm DL
 * `MaxIm DL Pro 7` - This is started automatically when we instantiate ASCOM object `MaxIm.Application` with the win32 COM client.  Must start after PlaneWave STI otherwise it cannot connect to the telescope mount interface without an app restart
 * `PlaneWave Interface v3` (PWI3) - This is started automatically inconified when we first instantiate the focuser through ASCOM (`ASCOM.PWI3.Focuser`)

### Start STI

Use full-path to launch `SitechExe.exe` - note you need to escape all backslashes for Python

Wait 5 seconds for it to launch and boot, then instantiate a `SiTech.Telescope` ASCOM instance.


In [14]:
#stiExe = r"C:\Program Files (x86)\Common Files\ASCOM\Telescope\SiTech\SitechExe.exe"

stiExe = r"SitechExe.exe" # if in path
os.startfile(stiExe)

time.sleep(5)

t0 = time.time()
SiTech = Dispatch("SiTech.Telescope")
dt = time.time() - t0
print(f'Dispatch() took {dt:.4f} seconds')
time.sleep(2)

SiTech.Connected = True

if (SiTech.Connected):
    print(f'Connected to {SiTech.Description}')
else:
    print(f'Connection attempt failed!')

Dispatch() took 2.0829 seconds
Connected to DEMONEXT SiTech Servo Controller in the OSU Lab


### Start MaxIm DL

Start the MaxIm DL `Application` and `CCDCamera` interfaces.


In [17]:
MaxIm = Dispatch("MaxIm.Application")
time.sleep(5)

MaxIm.LockApp = True
t0 = time.time()
MaxIm.TelescopeConnected = True
dt = time.time() - t0
print(f'MaxIm TelescopeConnected=True took {dt:.4f} seconds')

MaxCam = Dispatch("MaxIm.CCDCamera")
time.sleep(5)

t0 = time.time()
MaxCam.LinkEnabled = True
dt = time.time() - t0
print(f'MaxCam LinkEnabled took {dt:.4f} seconds')

# This makes sure MaxIm does not shutdown cameras if all COM objects terminate

MaxCam.DisableAutoShutdown = True

print('Done: MaxIm Startup Complete')

MaxIm TelescopeConnected=True took 0.0035 seconds
MaxCam LinkEnabled took 0.1548 seconds
Done: MaxIm Startup Complete


### Query CCD Camera Parameters

In [65]:
MaxCam = Dispatch("MaxIm.CCDCamera")
MaxCam.LinkEnabled = True
#time.sleep(5)

# print(f'Camera ID: {MaxCam.CameraName}')

print(f'Science Camera Parameters:')
print(f'  MaxIm DL Camera {MaxCam.MainCameraID}')
print(f'  Format: {MaxCam.CameraXSize}x{MaxCam.CameraYSize}')
print(f'  Pixel Size: {MaxCam.PixelSizeX:.1f}x{MaxCam.PixelSizeY:.1f} microns')
print(f'  Binning: {MaxCam.BinX}x{MaxCam.BinY}')
ccdXS = MaxCam.StartX
ccdXE = ccdXS + MaxCam.NumX
ccdYS = MaxCam.StartY
ccdYE = ccdYS + MaxCam.NumY
print(f'  Readout ROI: [{ccdXS}:{ccdXE},{ccdYS}:{ccdYE}]')

ccdStatus = MaxCam.CameraStatus
print(f'  CCD Camera Status: code {ccdStatus}: {camStatusCodes[ccdStatus]}')
print(f'  CCD Exposure Time: {MaxCam.ExposureTime} sec')

# on first read, camera can read a false number, give the camera FPGA time to spin up

ccdTemp = MaxCam.Temperature
if ccdTemp > 100.0: # still ramping up
    time.sleep(2)

print(f'\nCCD Thermoelectric Cooler Status:')
print(f'     Cooler State: {OnOff[MaxCam.CoolerOn]}, Cooler Power: {MaxCam.CoolerPower:.1f}%')
print(f'  CCD Temperature: {MaxCam.Temperature:.2f} C')
print(f' Base Temperature: {MaxCam.AmbientTemperature:.2f} C')
print(f' Cooler Set Point: {MaxCam.TemperatureSetpoint:.2f} C')

# Guide Camera

print(f'\nGuide Camera Parameters:')
print(f'  MaxIm DL Camera {MaxCam.GuiderCameraID}')
print(f'  Format: {MaxCam.GuiderXSize}x{MaxCam.GuiderYSize}')
print(f'  CMOS Temperature: {MaxCam.GuiderTemperature:.2f} C')

Science Camera Parameters:
  MaxIm DL Camera 1
  Format: 2048x2064
  Pixel Size: 15.0x15.0 microns
  Binning: 1x1
  Readout ROI: [0:2048,0:2064]
  CCD Camera Status: code 2: Camera idle
  CCD Exposure Time: 60.0 sec

CCD Thermoelectric Cooler Status:
     Cooler State: Off, Cooler Power: 10.0%
  CCD Temperature: 22.06 C
 Base Temperature: 21.88 C
 Cooler Set Point: -20.00 C

Guide Camera Parameters:
  MaxIm DL Camera 2
  Format: 1608x1104
  CMOS Temperature: 28.50 C


## Connect the SiTech Telescope Mount controller

Connect to the PlaneWave STI app's ASCOM interface `SiTech.Telescope` and query mount information.

In [43]:
SiTech = Dispatch("SiTech.Telescope")
SiTech.Connected = True

# Some driver info

if (SiTech.Connected):
    print(f'Connected to {SiTech.Description}')
    print(f'PlaneWave STI v{SiTech.DriverVersion}\n')
    
    print('Telescope Mount Status:')
    print(f'  RA/Dec Position: {SiTech.RightAscension:.5f} deg, {SiTech.Declination:.5f} deg')
    print(f'  Alt/Az Position: {SiTech.Altitude:.5f} deg, {SiTech.Azimuth:.5f} deg')
    print(f'    Sidereal Time: {SiTech.SiderealTime:.5f} hours')
   
    print(f'\nMount State:')
    print(f'  Tracking? {YesNo[SiTech.Tracking]}')
    print(f'   Slewing? {YesNo[SiTech.Slewing]}')
    print(f'   At Park? {YesNo[SiTech.AtPark]}')
    print(f'   At Home? {YesNo[SiTech.AtHome]}')
else:
    print(f'SiTech Telescope connection failed')


Connected to DEMONEXT SiTech Servo Controller in the OSU Lab
PlaneWave STI v1.4.3

Telescope Mount Status:
  RA/Dec Position: 17.42224 deg, -52.99554 deg
  Alt/Az Position: 5.33890 deg, 180.02726 deg
    Sidereal Time: 17.42527 hours

Mount State:
  Tracking? No
   Slewing? No
   At Park? Yes
   At Home? No


## Connect the PlaneWave Hedrick Focuser

The Hedrick Focuser at the CDK20 cassegrain focus is controlled using the PlaneWave `PWI3` app through
the `ASCOM.PWI3.Focuser` object.  Invoking this opens PWI3 minimized on the task bar.

We also start the PlaneWave AutoFocus application. Although it cannot be tested except on-sky, we
can verify it is available and correctly installed.

In [46]:
Focuser = Dispatch("ASCOM.PWI3.Focuser")
AutoFoc = Dispatch("PlaneWave.AutoFocus")

time.sleep(1)

t0 = time.time()
Focuser.Connected = True
dt = time.time()-t0

if Focuser.Connected:
    print(f'{Focuser.Description} connected in {dt:.3f} sec')
    print(f'  Focuser Linked? {YesNo[Focuser.Link]}')
    time.sleep(1)
    print(f'  Focuser Position: {Focuser.Position} microns')
    print(f'  Focuser Moving? {YesNo[Focuser.IsMoving]}')
else:
    print(f'Focuser not connected')


PlaneWave Focuser (PWI3) connected [2.079 sec]
  Focuser Linked? Yes
  Focuser Position: 2000 microns
  Focuser Moving? No


## End of Startup

Stop with this cell, beyond is moving things...

**Shutdown process is in the last few cells**

## Utility functions for the telescope and instrument

### getTelInfo(tel)

#### Arguments
 * tel - `SiTech.Telescope` ASCOM instance

#### Description

Query the telescope and return info needed to augment the default FITS headers as a dictionary.  Keywords
are the verbatim FITS header keywords, values queried from `tel` object proprties, where
`tel` is a `SiTech.Telescope` ASCOM instance.

We compute telescope hour angle (`TelHA`) and $\sec(Z_t)$ (`SecZ`) from telescope position information
in decimal degrees.

#### Note: Airmass vs. Sec(ZD)

We compute SECZ (secant of zenith distance) but do not call it airmass as strictly 
speaking airmass is only approximately sec(ZD) for small zenith distance angles. 
At larger ZD closer to the horizon the assumption of a geometric plane-parallel atmosphere
embodied by the sec(ZD) approximation fails badly, and in practice we resort to interpolative
models that take account of the departure of the light path from the simple geometric
approximation due to atmospheric refraction (e.g., the traditional Young & Irvine 1968 formula).
More recently some observers are resorting to more sophisticated model atmospheres methods
for sight lines getting within 10-degrees of the horizon.

In practice DEMONEXT won't work at large zenith distances, but better to put a basic
observable in the header and worry about more sophisticated corrective calculation offline
and not in the critical path for the time-sensitive data acquisition.  SECZ has the virtue
of being a good-enough proxy for airmass.

In [50]:
def getTelInfo(tel):
    telAz = tel.Azimuth
    telAlt = tel.Altitude

    # estimate sec(Z) from altitude (this is sec(Z_true) ignoring refraction)
    
    if telAlt < 1.0:
        SecZ = 99.99
    else:
        SecZ = 1.0/math.cos(math.radians(90.0-telAlt))

    # compute telescope hour angle, overrides whatever MaxIm is computing
    
    telHA = tel.SiderealTime - tel.RightAscension

    # return a dictionary with the extra telescope info
    
    return {'TELALT':telAlt,
            'TELAZ':telAz,
            'TELHA':telHA,
            'SECZ':SecZ}


### getInstStatus(cam,foc)

#### Arguments
 * `cam` is a `MaxIm.CCDCamera` ASCOM instance
 * `foc` is a `ASCOM.PWI3.Focuser` ASCOM instance

#### Description

Query the instrument system and return instrument status info needed to augment the default FITS
headers as a dictionary. Keywords are the verbatim FITS header keywords, values queried from 
`cam` and `foc` ASCOM object properties we want to query.

In [53]:
def getInstStatus(cam,foc):
    
    # CCD Cooler information

    ccdTemp = cam.Temperature
    if cam.CoolerOn:
        tecState = 'ON'
    else:
        tecState = 'OFF'
    tecSetPoint = cam.TemperatureSetpoint
    tecPower = cam.CoolerPower
    baseTemp = cam.AmbientTemperature # really the TEC base temperature

    # Focuser information

    focPos = foc.Position
    mirrorTemp = foc.Temperature # EFA reports mirror temp sensor but does not expose ambient temp sensor

    return {'CCDTEMP':ccdTemp,
            'TECSTATE':tecState,
            'TECSETPT':tecSetPoint,
            'TECPOWER':tecPower,
            'TECTEMP':baseTemp,
            'FOCUSPOS':focPos,
            'MIRROR_T':mirrorTemp}

### updateFITSHeader(cam, fitsInfo)

#### Arguments:

 * `cam` - `MaxIm.CCDCamera` ASCOM instance
 * `fitsInfo` - dictionary of keyword/value pairs with FITS header entries

#### Description:

A function to upload supplmentary metadata for DEMONEXT FITS headers to MaxIm using the `MaxIm.CCDCamera.SetFITSKey()` method.  The supplementary metadata is in a fitsInfo
dictionary populated with fixed and programmatic keywords (see the master `fitsInfo`
dictionary definition above).

In [56]:
# Add info to the headers 

def updateFITSHeader(cam, fitsInfo):
    for key,value in fitsInfo.items():
        try:
            cam.setFITSKey(key,value)
        except Exception as e:
            print(f"FITS keyword '{key:8s} = {value}' raised exception",e)


# System Functional Tests

Cells below test various functions of the system.  It assumes all observatory
systems are running and connected

## Move the Hedrick Focuser

Set the Hedrick focuser to an absolute target focus in microns. The `Focuser.Move()` method is non-blocking,
so we execute it then watch the `Focuser.IsMoving` boolean until it stops moving, then readout
the position.

Set `queryCadence` to monitor position. 0.05 sec about fastest, 0.1 sec works fine

In [42]:
targetFocus = 2000
queryCadence = 0.1

print(f'Focuser starting at {Focuser.Position} microns, moving to {targetFocus} microns')

t0 = time.time()
Focuser.Move(targetFocus)
while(Focuser.IsMoving):
    sys.stdout.write(f'\r  Focuser Position: {Focuser.Position:5d} microns')
    sys.stdout.flush()
    time.sleep(queryCadence)
dt = time.time()-t0-queryCadence
print(f'\nDone: Focuser at {Focuser.Position} microns, move time {dt:.3f} seconds')

Focuser starting at 1000 microns, moving to 2000 microns
  Focuser Position:  2000 microns
Done: Focuser at 2000 microns, move time 14.483 seconds


# Move the telescope

**Only do this if you are next to the telescope so you can hit the STOP button if needed**

(this is why you were warned not to run all cells blindly - bad things could happen)

## Home the Telescope

This cell connects to the STI app, checks park status, then instructs the telescope to the home position,
which is at HA=0, Dec=0.  

Unlike other telescope moves `SiTech.FindHome` is a **blocking** method that waits until Home is reached or
its fails.

In [45]:
SiTech = Dispatch("SiTech.Telescope")
SiTech.Connected = True

if SiTech.AtPark:
    print(f'Unparking the telescope...')
    SiTech.UnPark
else:
    print(f'Telecope is not parked.')

time.sleep(2)

t0 = time.time()
print(f'Homing the telescope...')
SiTech.FindHome
dt = time.time()-t0
time.sleep(2.0)

if (SiTech.AtHome):
    print(f'Done: Telescope is at home - time to home {dt:.2f} seconds')
else:
    print(f'Error: Telescope not at home after {dt:.2f} seconds')

Unparking the telescope...
Homing the telescope...
Done: Telescope is at home - time to home 36.05 seconds


## Park the Telescope

Unlike homing, parking is not a blocking operation, so we have to watch the `SiTech.Slewing` property to change
from True to False.  A 1 second sleep cadence seems sufficient.

If you don't want for at least the slew settling time, you might check `SiTech.AtPark` before motion has finished
and reach parked. In tests we got no `AtPark` assert if we didn't wait long enough.

In [53]:
queryCadence = 0.1 # seconds

if SiTech.AtPark:
    print('Telescope is parked')
    print(f'Park Alt={SiTech.Altitude:.5f} deg Az={SiTech.Azimuth:.5f} deg')
else:
    print('Parking the telescope...')
    t0 = time.time()
    SiTech.Park
    while(SiTech.Slewing):
        time.sleep(queryCadence)
        sys.stdout.write(f'\r  Telescope at Alt={SiTech.Altitude:.5f} deg Az={SiTech.Azimuth:.5f} deg    ')
        sys.stdout.flush()
        
    time.sleep(SiTech.SlewSettleTime)
    dt = time.time()-t0-queryCadence-SiTech.SlewSettleTime
    print(f'\nDone: execution time {dt:.2f} seconds')

    if SiTech.AtPark:
        print('Telescope is parked')
        print(f'Park Position: Alt={SiTech.Altitude:.5f} deg Az={SiTech.Azimuth:.5f} deg')
    else:
        print('Telescope park attempt failed')
        print(f'Telescope at Alt={SiTech.Altitude:.5f} deg Az={SiTech.Azimuth:.5f} deg')
     

Parking the telescope...
  Telescope at Alt=4.99810 deg Az=180.02838 deg     
Done: execution time 28.64 seconds
Telescope is parked
Park Position: Alt=4.99806 deg Az=180.02838 deg


## Get filter table from MaxIm DL

Read and print the filter table, verifying the MaxIm app we are connected to has the
correct filter table. This has to be edited using the MaxIm app at the time the configuration
is established; they provide no methods for changing the filter table by MaxIm script clients.

In [48]:
MaxCam = Dispatch("MaxIm.CCDCamera")
MaxCam.LinkEnabled = True

print(f'Filter Position: Slot {MaxCam.Filter}, FilterName: {MaxCam.FilterNames[MaxCam.Filter]}')

filterList = MaxCam.FilterNames
print(f'Pos  Filter')
for i in range(len(filterList)):
    print(f' {i}   {filterList[i]}')
   

Filter Position: Slot 4, FilterName: g_SDSS
Pos  Filter
 0   B_Bessel
 1   V_Bessel
 2   R_Bessel
 3   I_Bessel
 4   g_SDSS
 5   r_SDSS
 6   i_SDSS
 7   z_SDSS
 8   Clear
 9   Open


## Take an exposure with the Science Camera

Acquire an image, science, dark, or bias using the `CCDCamera.Expose()` method, with an option to save 
as FITS with an augmented header. Operation includes setting the filter if acquiring a science image.

 * **Science** images are exposed for a non-zero expTime and you specify the filter, imgType=1 (open shutter)
 * **Dark** images are acquired for a non-zero expTime and no filter is specified, imgType=0 (closed shutter)
 * **Bias** images are 0s dark images and no filter is specified, imgType=0 (closed shutter)

Set `saveImage = True` to save the image as FITS format.  We augment the header with extra information,
simulating adding site, project, instrument, and additional telescope info (Alt, Az, SecZ aka "Airmass").

This sandbox version times the various steps so we can get a quantitative breakdown of the different
non-exposure overheads.

During the exposure, we watch two `MaxIm.CCDCamera` properties:
 * `CCDCamera.ImageReady` - True if an image is ready in the image buffer, False if not ready
 * `CCDCamera.CameraStatus` - current status of the camera, using `camStatusCodes` to translate codes to string.

We only print the status message when the `CCDCamera.CameraStatus` propertie changes state.

In [60]:
# General image procedure 

#dataDir = r"C:\Users\DEMONEXT\Documents\RawData"
dataDir = r"D:\Data"
outFile = r"sciImage.fits"

objectID = 'Test Image'

saveImage = True

expTime = 60.0 # sec, 0.0 sec is a bias image
imgType = 1   # 1=science (open shutteR), 0=dark or bias (closed shutter)
imgFilt = 7   # select filter by number 0..9

filterList = MaxCam.FilterNames

# start the timer

t0 = time.time()

# make sure we are attached and ready (part of the overhead)

MaxCam = Dispatch("MaxIm.CCDCamera")
MaxCam.LinkEnabled = True

if imgType:
    print(f"Starting {expTime:.1f} sec Science image with the {filterList[imgFilt]} filter...")
else:
    if expTime > 0:
        print(f'Starting {expTime:.1f} sec Dark image...')
    else:
        print(f'Starting Bias image...')

# start the timer

dt = time.time() - t0

# last camera state

lastState = MaxCam.CameraStatus

# Create a new FITS header keyword dictionary from the master and populate with info specific to this image
# starting with object info and filter wheel position if a science image (not recorded by MaxIm)

newHeader = fitsHeader
newHeader['OBJECT'] = objectID
if imgType > 0:
    newHeader['FILTPOS'] = imgFilt

# start of exposure instrument and telescope data snapshots

newHeader.update(getInstStatus(MaxCam,Focuser))
newHeader.update(getTelInfo(SiTech))

# start the exposure

if imgType:
    MaxCam.Expose(expTime,imgType,imgFilt)
else:
    MaxCam.Expose(expTime,imgType)
    
while(not MaxCam.ImageReady):
    camStatus = MaxCam.CameraStatus
    if lastState is not camStatus:
        dt = time.time()-t0
        print(f'{dt:8.4f}s: Status {camStatus:2d} {camStatusCodes[camStatus]}')
        lastState = camStatus

execTime= time.time()-t0
if imgType:
    print(f'  ExpDone: {expTime:.1f} sec {filterList[imgFilt]} image ready in buffer, execTime = {execTime:.3f} seconds')
else:
    if expTime > 0:
        print(f'  ExpDone: {expTime:.1f} sec Dark image ready in buffer, execTime = {execTime:.3f} seconds')
    else:
        print(f'  ExpDone: Bias image ready in buffer, execTime = {execTime:.3f} seconds')

if saveImage:
    outName = fr"{dataDir}\{outFile}"

    t0 = time.time()

    # update FITS headers

    updateFITSHeader(MaxCam,newHeader)

    dtf = time.time()-t0
    print(f'  FITS header update took {dtf:.3f} seconds')

    # write the filename

    if MaxCam.SaveImage(outName):
        dt = time.time()-t0
        print(f'  Wrote {outName} in {dt-dtf:.3f} seconds')
        print(f'  Total update+write time: {dt:.3f} seconds')
    else:
        dt = time.time()-t0
        print(f'** Image write with CCDCamera.SaveImage() failed')

    execTime = execTime + dt
else:
    print(f'** Note: image not saved')

# All done

if expTime > 0:
    overHead = execTime - expTime
    print(f'DONE: Execution time: {execTime:.3f} seconds, non-exposure overhead: {overHead:.3f} seconds')
else:
    print(f'DONE: Execution time: {execTime:.3f} seconds')

SyntaxError: unterminated string literal (detected at line 4) (3107239050.py, line 4)

## Evaluation: Which way to create FITS images?

### Method 1: Retrieve raw data array and write FITS with astropy.io.fits

Get the image data using `MaxCam.ImageArray`, convert to numpy array object, then create
a basic FITS header and write using `astropy.io.fits` methods.

Check timing

In [ ]:
#dataPath = r"C:\Users\DEMONEXT\Documents\RawData"
dataPath = r"D:\Data"
fileName = "biasAstroPy.fits"

t0 = time.time()
rawImg = np.array(MaxCam.ImageArray)
dt = time.time()-t0
print(f'Raw Image data transfer took {dt:.3f} seconds')
t0 = time.time()
hdu = fits.PrimaryHDU(data=rawImg)
hdul = fits.HDUList([hdu])
hdul.writeto(fr'{dataPath}\{fileName}',overwrite=True)
dt = time.time() - t0
print(fr'Wrote {dataPath}\{fileName} in {dt:.3f} seconds with astropy.io.fits.writeto()')

### Method 2: Write FITS direct from MaxIM using CCDCamera.SaveImage() method

Write the FITS data using the `CCDCamera.SaveImage()` method.  This creates a FITS
file with header info from MaxIm that includes all relevant metadata it knows.

We add supplementary metadata to the FITS headers using our `updateFits()` function.

Time this method, compare to astropy above, including the extra steps to populate the MaxIm
FITS header buffer.

In [ ]:
# write to different filename using MaxIm DL CCDCamera.SaveImage() method

t0 = time.time()

# requires newHeader defined above 

updateFITSHeaders(MaxCam,newHeader)

outName = fr"C:\Users\DEMONEXT\Documents\RawData\biasMaxIm.fits"
if MaxCam.SaveImage(outName):
    dt = time.time()-t0
    print(f'Wrote {outName} in {dt:.3f} seconds with CCDCamera.SaveImage()')
else:
    print(f'Image write with CCDCamera.SaveImage() failed')

### Analysis

Overhead for writing directly from MaxIm is 0.05 seconds on average.

Overhead for downloading the image array and creating FITS using astropy is 0.75 seconds on average, of which
0.73 seconds is required just to transfer the image from MaxIm's memory buffer to the notebook with the
`CCDCamera.ImageArray` property.  After that average FITS write time is ~0.03-0.04 seconds.

Using the `updateFITSHEader()` function to update the headers, the MaxIm write overhead went up to 0.79sec,
comparable to the `astrop.io.fits` methods, but we get a richer FITS header from MaxIm with a more
accurate shutter-open time.

### Conclusion

 > Best to use the native `CCDCamera.SaveImage()` method.  The FITS header already has most but not all of what we need

## CCD Camera Cooler Control

These cells are workign out how to set the CCD thermoelectric cooler (TEC).

The FLI ProLine camera has a 3-stage Peltier TEC that can cool the camera
to about -55C below ambient. However, the FLI camera has a fairly sophisticated
PID (Proportional-Integral-Differential) thermal control system that can hold
the temperature at a given **absolute** temperature that does not float relative
to ambient air provided $\left|\Delta T\right|$ is less than $\sim$50C.

Operationally, this means for a reasonable observing site we can choose a cooler
setpoint of $-20$C and expect it to hold.  In the lab it might be warmer, and
we'd see the camera struggle to keep up.

### MaxIm.CCDCamera properties

CCDCamera object properties related to the CCD camera TEC control are
 * `CCDCamera.Temperature` - float, the current CCD detector temperature in Celsius
 * `CCDCamera.AmbientTemperature` - float, the current temperature of the TEC heat sink (tied to ambient, but likely warmer)
 * `CCDCamera.CoolerOn` - boolean, True = TEC is powered on and actively cooling, False = TEC powered off
 * `CCDCamera.CoolerPower` - int, percent cooler power if `CoolerOn=True`
 * `CCDCamera.TemperatureSetpoint` - float, target CCD sensor temperature in Celsius

TEC cooler control in the FLI is non-blocking (asynchronous), when you set `CoolerOn=True` and a setpoint
with `TemperatureSetpoint`, it takes time to reach and then hold that temperature.

Warm-up should be done in a controlled fashion, usually by setting the setpoint temperature to the
ambient temperature and judging when it gets within 1-2C of the target, at which point you can
turn the TEC off (`CoolerOn=False`). Just turning the cooler off will make an uncontrolled warm
up which is probably benign, but the idea is to do it in a controlled fashion to avoid thermal
shocks to the sensor which can led to damage.

The two following cells are developing cool-down and warm-up procedures.  We also time the process
to get an idea of how long it takes.

For safety we set a maximum cooling time for the process of 2x the expected cooling time, which is
about 10 minutes to go from about 25C in the lab to -20C operating temperature.  If the set point
temperature has not been reached in `maxCoolTime`, we raise an error condition.

Warm-up is usually shorter about 5-6 minutes, so set the warm-up time to also about 2x

### CCD Cool-Down Procedure

In [33]:
ccdSetPoint = -20.0 # desired operating temp in Celsius
tempTolerance = 0.2 # tolerance in C for declaring "at temp"
queryCadence = 3.0 # query every 1s during temperature ramp-down

maxCoolTime = 20.0*60.0 # maximum cooling time set to 20 minutes = 1200 seconds.

# Make sure we are connected to the cameras

MaxCam = Dispatch("MaxIm.CCDCamera")
MaxCam.LinkEnabled = True

# Where are we now

print(f'CCD TEC is {OnOff[MaxCam.CoolerOn]}:')
print(f'  SetPoint={MaxCam.TemperatureSetpoint:.1f}C')
print(f'  TECPower={MaxCam.CoolerPower}%')
print(f'   CCDTemp={MaxCam.Temperature:.2f}C')
print(f'  BaseTemp={MaxCam.AmbientTemperature:.2f}C') # "ambient" is really the base of the TEC at the heat sink

# Start cooldown cycle

print(f'\nStarting cooldown to {ccdSetPoint:.1f}C\n')
t0 = time.time()
MaxCam.TemperatureSetpoint = ccdSetPoint
MaxCam.CoolerOn = True
lastCT = MaxCam.Temperature
deltaT = abs(lastCT - ccdSetPoint)
lastTime = time.time() - t0

cdTime = 0.0 # initialize the cool-down timer

# show progress as we ramp toward operating temperature

print('             Time    CCD     Base   Power   Rate')
while (deltaT > tempTolerance and cdTime < maxCoolTime):
    nowCT = MaxCam.Temperature
    nowCP = MaxCam.CoolerPower
    nowBT = MaxCam.AmbientTemperature
    deltaT = abs(nowCT - ccdSetPoint)
    cdTime = time.time()-t0

    dt = cdTime - lastTime
    if dt > 0.0:
        cdRate = (nowCT - lastCT)/dt
    else:
        cdRate = (nowCT - lastCT)/queryCadence
        
    sys.stdout.write(f'\r Cooling: {cdTime:7.1f}s {nowCT:6.2f}C {nowBT:6.2f}C  {nowCP:3d}%  {cdRate:6.3f} C/sec  ')
    sys.stdout.flush()
    lastTime = cdTime
    lastCT = nowCT
    time.sleep(queryCadence)
    
# either reached the set point or timed out

cdTime = time.time()-t0
if cdTime >= maxCoolTime:
    print(f'\n\n: WARNING: CCD did not reach set point of {ccdSetPoint:.1f}C after {maxCoolTime/60.0:.1f} minutes')
else:
    print(f'\n\nDone: CCD at {ccdSetPoint:.1f}C, cool-down time = {cdTime:.1f} seconds')

CCD TEC is Off:
  SetPoint=-20.0C
  TECPower=9%
   CCDTemp=22.38C
  BaseTemp=22.25C

Starting cooldown to -20.0C

             Time    CCD     Base   Power   Rate
 Cooling:   538.6s -19.88C  29.62C   90%  -0.042 C/sec  

Done: CCD at -20.0C, cool-down time = 541.6 seconds


### Warm-up to ambient

Here we make the set-point temperature the ambient temperature of the camera, but
allow "done" to be within 5C of the target.  This is a safe temperature at which
to power off the cooler and allow it to ramp to ambient uncontrolled.

In [54]:
queryCadence = 3.0 # query every 1s during temperature ramp-up
maxWarmTime = 10.0*60.0 # maximum warm-up time in seconds

# Make sure we are connected to the cameras

MaxCam = Dispatch("MaxIm.CCDCamera")
MaxCam.LinkEnabled = True

# New setpoint is ambient, tolerance is within 5C, safe to coast to temp.

ccdSetPoint = MaxCam.AmbientTemperature - 10.0 # allows for warm heat sink
tempTolerance = 2.0 # C

# Where are we now?

print(f'CCD TEC is {OnOff[MaxCam.CoolerOn]}:')
print(f'  SetPoint={MaxCam.TemperatureSetpoint:.1f}C')
print(f'  TECPower={MaxCam.CoolerPower}%')
print(f'   CCDTemp={MaxCam.Temperature:.2f}C')
print(f'  BaseTemp={MaxCam.AmbientTemperature:.2f}C')

# Start cooldown cycle

print(f'\nStarting warm-up to {ccdSetPoint:.1f}C\n')
t0 = time.time()
MaxCam.TemperatureSetpoint = ccdSetPoint
MaxCam.CoolerOn = True
lastCT = MaxCam.Temperature
deltaT = abs(lastCT - ccdSetPoint)
lastTime = time.time() - t0

# initialize the warm-up timer

wuTime = 0.0

# show warm-up progress

print('             Time    CCD     Base   Power   Rate')
while (deltaT > tempTolerance and wuTime < maxWarmTime):
    nowCT = MaxCam.Temperature
    nowCP = MaxCam.CoolerPower
    nowBT = MaxCam.AmbientTemperature
    deltaT = abs(nowCT - ccdSetPoint)
    wuTime = time.time()-t0

    dt = wuTime - lastTime
    if dt > 0.0:
        wuRate = (nowCT - lastCT)/dt
    else:
        wuRate = (nowCT - lastCT)/queryCadence
        
    sys.stdout.write(f'\r Warming: {wuTime:7.1f}s {nowCT:6.2f}C {nowBT:6.2f}C  {nowCP:3d}%  {wuRate:6.3f} C/sec  ')
    sys.stdout.flush()
    time.sleep(queryCadence)

# either reached the warm-up set point or timed out

wuTime = time.time()-t0
if wuTime >= maxWarmTime:
    print(f'\n\nWarning: CCD did not reach ambient after {maxWarmTime/60:.1f} minutes')
else:
    print(f'\n\nDone: CCD at {ccdSetPoint:.1f}C, warm-up time = {wuTime:.1f} seconds')

# power off cooler

MaxCam.CoolerOn = False
print('     CCD Cooler OFF')

CCD TEC is On:
  SetPoint=-20.0C
  TECPower=78%
   CCDTemp=-20.00C
  BaseTemp=29.62C

Starting warm-up to 19.6C

             Time    CCD     Base   Power   Rate
 Warming:   481.9s  17.69C  23.75C    0%   0.078 C/sec  

Done: CCD at 19.6C, warm-up time = 484.9 seconds
     CCD Cooler OFF


## Guide Camera

Connect with the guide camera, take an image, capture the image array, and store the image as
a simple FITS file using astropy (no method in MaxIm to save guider images).

Here we also have a little fun coding a countdown and readout indicator (mostly psychological since
we don't get granular info about guide camera control status like we do with the CCD camera).

In [ ]:
queryCadence = 0.2

# Guide Camera status

print(f'\nGuide Camera Status:')
print(f'  MaxIm DL Camera {MaxCam.GuiderCameraID}')
print(f'  Format: {MaxCam.GuiderXSize}x{MaxCam.GuiderYSize}')
print(f'  CMOS Temperature: {MaxCam.GuiderTemperature:.2f} C')
print(f'  Guider Running? {YesNo[MaxCam.GuiderRunning]}')
print(f'   Guider Moving? {YesNo[MaxCam.GuiderMoving]}')
print(f'  Guider Calibration State: {guiderCalStates[MaxCam.GuiderCalState]}')

# Try to take a 10s guider exposure

guiderExpTime = 5.0

t0 = time.time()
MaxCam.GuiderExpose(guiderExpTime)
tStart = time.time()

readoutStr = ' Readout: '
readout = False
while(MaxCam.GuiderRunning):
    time.sleep(queryCadence)
    dt = time.time() - tStart
    tLeft = guiderExpTime - dt
    if tLeft > 0:
        sys.stdout.write(f'\r Exposing: {tLeft:4.1f}s of {guiderExpTime:3.1f}s remaining  ')
        sys.stdout.flush()
    else:
        if not readout:
            tLeft = 0.0
            sys.stdout.write(f'\r Exposing: {tLeft:4.1f}s of {guiderExpTime:3.1f}s remaining  \n')
            sys.stdout.flush()
            readout = True
            readoutStr += '-'
            sys.stdout.write(f'\r {readoutStr}')
        else:
            readoutStr += '-'
            sys.stdout.write(f'\r {readoutStr}')
    sys.stdout.flush()

# cleanup

execTime = time.time()-t0
print(f'\nDone: Guide camera exposure completed, execTime = {execTime:.3f} seconds')

### Capture and save the guider image as FITS

MaxIm does not provide and equivalent of the SaveImage() method for the guide
camera, so we capture the guide frame with `CCDCamera.GuideArray` and use `astropy.io.fits` methods
to create a basic FITS header.

In [ ]:
#dataPath = r"C:\Users\DEMONEXT\Documents\RawData"
dataPath = r"D:\Data"
fileName = "guiderImage.fits"

t0 = time.time()
rawImg = np.array(MaxCam.GuiderArray)
dt = time.time()-t0
print(f'Raw guider image transfer took {dt:.3f} seconds')
t0 = time.time()
hdu = fits.PrimaryHDU(data=rawImg)

# add keywords to the basic header

hdul = fits.HDUList([hdu])
hdr = hdul[0].header
hdr['INSTRUME'] = 'DEMONEXT Guide Camera'
hdr['GUIDECAM'] = ('ZWO ASI432MM','Guide Camera Model')
hdr['EXPTIME'] = (guiderExpTime,'Exposure Time [sec]')
hdr['GCAMTEMP'] = (MaxCam.GuiderTemperature,'CMOS sensor temperature [C]')

# save, overwrite enabled

hdul.writeto(fr'{dataPath}\{fileName}',overwrite=True)
dt = time.time() - t0
print(fr'Wrote {dataPath}\{fileName} in {dt:.3f} seconds with astropy.io.fits.writeto()')

## End of Functional Test Cells

Beyond this point is the system shutdown procedure.

# System Shutdown

Need to make sure ASCOM instances are cleanly detached, or you can get stuck processes
that lead to strange errors ("why is it not connecting? It was just connected?")


In [60]:
print('Disconnecting Hedrick Focuser (PWI3)')
try:
    Focuser.Connected = False
except:
    print('')
        
print('Disconnecting Telescope from MaxIm')
try:
    MaxIm.TelescopeConnected = False
    time.sleep(2)
except:
    print('')
    
print('Disconnecting Cameras from MaxIm')
try:
    MaxCam.LinkEnabled = False
    time.sleep(2)
    MaxIm.LockApp = False
except:
    print('')
    
print('Disconnecting SiTech Mount Controller (STI)')
try:
    SiTech.Connected = False
    time.sleep(5)
except:
    print('')

print('Done: DEMONEXT subsystems ASCOM links disconnected.')


Disconnecting Hedrick Focuser (PWI3)
Disconnecting Telescope from MaxIm
Disconnecting Cameras from MaxIm
Disconnecting SiTech Mount Controller (STI)
Done: DEMONEXT subsystems ASCOM links disconnected.


## Take down Apps running on the system

In [63]:
print('shutting down MaxIm DL app...')
result = os.system("taskkill /f /im MaxIm_DL.exe /t")

print('shutting down PWI3 app...')
result = os.system("taskkill /f /im PWI3.exe /t")

print('shutting down PlaneWave STI app...')
result = os.system("taskkill /f /im SitechExe.exe /t")

print('Done: all apps shutdown')

shutting down MaxIm DL app...
shutting down PWI3 app...
shutting down PlaneWave STI app...
Done: all apps shutdown
